In [0]:
# ===== CONFIG (edit these BEFORE running) =====
CATALOG = "your_catalog"         # example: demo_catalog
SCHEMA = "your_schema"           # example: idp_demo
DOCS_LOCAL_PATH = "data/sample_docs"   # relative path in repo
DOCS_TARGET_DBFS = "dbfs:/FileStore/idp-demo/docs/"  # where to copy in workspace
# ==============================================

In [0]:
%sql
select *
from read_files('/Volumes/idp_demo/idp_bronze/pdfs')

In [0]:
%sql
create or replace table idp_demo.idp_bronze.parsed_data as
select path,
ai_parse_document(content) as parsed_content
from read_files('/Volumes/idp_demo/idp_bronze/pdfs')

In [0]:
%sql
select * from idp_demo.idp_bronze.parsed_data

In [0]:
%sql
create schema if not exists idp_demo.idp_silver

In [0]:
%sql
create or replace table  idp_demo.idp_silver.better_data as 
SELECT 
  path,
  concat_ws('\n',
    transform(
      try_cast(parsed_content:document:elements AS ARRAY<VARIANT>),
      e -> coalesce(try_cast(e:content AS STRING), '')
    )
  ) AS doc_text
FROM negin_demo.idp.parsed_data;

In [0]:
%sql
select * from idp_demo.idp_silver.better_data

In [0]:

%sql
create or replace table idp_demo.idp_silver.classified_data as
select *,
ai_classify(doc_text, array('construction_invoice','apartment_rental','purchase_order')) as document_classification
 from negin_demo.idp.better_data

In [0]:
%sql
select * from idp_demo.idp_silver.classified_data

In [0]:
%sql select * from idp_demo.idp_silver.classified_data
where document_classification = 'purchase_order'


In [0]:
%sql
select *, ai_extract(doc_text, array('po_number','po_date','buyer','vendor','item_code','description','unit_price','quantity','total')) as extracted from idp_demo.idp_silver.classified_data
where document_classification ='purchase_order'

In [0]:
%sql
select *,
ai_extract(
  doc_text,
  array(
    'purchase order number',
    'purchase order date',
    'buyer company name',
    'vendor company name',
    'item codes from the line items',
    'product descriptions from the line items',
    'unit price for each item',
    'quantity for each item',
    'total amount of the purchase order'
  )
) as extracted
from idp_demo.idp_silver.classified_data
where document_classification = 'purchase_order'

In [0]:
%sql
create or replace table idp_demo.idp_silver.purchase_orders as
select *,
ai_extract(
  doc_text,
  array(
    'purchase_order_number',
    'purchase_order_date',
    'buyer_company_name',
    'vendor_company_name',
    'item_codes_from_the_line_items',
    'product_descriptions_from_the_line_items',
    'unit_price_for_each_item',
    'quantity_for_each_item',
    'total_amount_of_the_purchase_order'
  )
) as extracted
from idp_demo.idp_silver.classified_data
where document_classification = 'purchase_order'

In [0]:
%sql
select * from idp_demo.idp_silver.purchase_orders

In [0]:
%sql
create schema if not exists idp_demo.idp_gold

In [0]:
%sql
create or replace table idp_demo.idp_gold.purchase_orders as 
select 
path , 
extracted.purchase_order_number as po_number,
extracted.purchase_order_date as po_date,
extracted.buyer_company_name as buyer,
extracted.vendor_company_name as vendor,
extracted.item_codes_from_the_line_items as item_code,
extracted.product_descriptions_from_the_line_items as description,
extracted.unit_price_for_each_item as unit_price,
extracted.quantity_for_each_item as quantity,
extracted.total_amount_of_the_purchase_order as total
from idp_demo.idp_silver.purchase_orders

In [0]:
%sql
select * from idp_demo.idp_gold.purchase_orders